In [0]:
WITH std AS (
  SELECT total_columns AS standard_column_count
  FROM metadata_governance.gold.table_governance_summary
  GROUP BY total_columns
  ORDER BY COUNT(*) DESC, total_columns DESC
  LIMIT 1
),
tbl AS (
  SELECT
    COUNT(*)                                  AS total_tables,
    SUM(s.total_columns)                      AS total_columns,
    ROUND(AVG(s.table_completeness_pct), 1)   AS avg_table_completeness_pct,
    SUM(s.pii_non_compliant_count)            AS total_pii_noncompliant,
    COUNT(DISTINCT s.system_name)             AS total_source_systems,
    MAX(st.standard_column_count)             AS standard_column_count,
    ROUND(100.0 * SUM(CASE WHEN s.total_columns = st.standard_column_count THEN 1 ELSE 0 END)
          / COUNT(*), 1)                      AS pct_structurally_compliant
  FROM metadata_governance.gold.table_governance_summary s
  CROSS JOIN std st
),
colm AS (
  SELECT
    ROUND(100.0 * SUM(CASE WHEN unowned     THEN 0 ELSE 1 END) / COUNT(*), 1) AS pct_stewardship_coverage,
    ROUND(100.0 * SUM(CASE WHEN uncertified THEN 0 ELSE 1 END) / COUNT(*), 1) AS pct_certified_any_level
  FROM metadata_governance.gold.column_governance_detail
)
SELECT tbl.*, colm.*
FROM tbl CROSS JOIN colm;

In [0]:
SELECT
  maturity_tier,
  COUNT(*)                                           AS table_count,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_tables,
  ROUND(AVG(table_completeness_pct), 2)              AS avg_completeness_pct,
  CASE maturity_tier WHEN 'High' THEN 1 WHEN 'Medium' THEN 2 ELSE 3 END AS tier_order
FROM metadata_governance.gold.table_governance_summary
GROUP BY maturity_tier
ORDER BY tier_order;

In [0]:
WITH std AS (
  SELECT total_columns AS standard_column_count
  FROM metadata_governance.gold.table_governance_summary
  GROUP BY total_columns
  ORDER BY COUNT(*) DESC, total_columns DESC
  LIMIT 1
),
flagged AS (
  SELECT
    CASE WHEN s.total_columns = st.standard_column_count
         THEN 'Meets column standard'
         ELSE 'Does not meet standard' END AS structural_status
  FROM metadata_governance.gold.table_governance_summary s
  CROSS JOIN std st
)
SELECT
  structural_status,
  COUNT(*)                                           AS table_count,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_tables
FROM flagged
GROUP BY structural_status
ORDER BY table_count DESC;

In [0]:
SELECT
  total_columns AS columns_in_table,
  COUNT(*)      AS number_of_tables
FROM metadata_governance.gold.table_governance_summary
GROUP BY total_columns
ORDER BY columns_in_table;

In [0]:
SELECT * FROM (
  SELECT 'Data steward (owner)' AS governance_field,
         ROUND(100.0 * SUM(CASE WHEN unowned THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_missing
  FROM metadata_governance.gold.column_governance_detail
  UNION ALL
  SELECT 'Certification level',
         ROUND(100.0 * SUM(CASE WHEN uncertified THEN 1 ELSE 0 END) / COUNT(*), 1)
  FROM metadata_governance.gold.column_governance_detail
) AS gaps
ORDER BY pct_missing DESC;

In [0]:
SELECT
  CASE WHEN uncertified THEN 'Uncertified' ELSE 'Certified (any level)' END AS certification_status,
  COUNT(*)                                           AS column_record_count,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_records
FROM metadata_governance.gold.column_governance_detail
GROUP BY uncertified
ORDER BY column_record_count DESC;

In [0]:
SELECT
  system_name,
  COUNT(*)                              AS table_count,
  ROUND(AVG(table_completeness_pct), 1) AS avg_completeness_pct,
  SUM(pii_non_compliant_count)          AS pii_risk_count
FROM metadata_governance.gold.table_governance_summary
GROUP BY system_name
ORDER BY avg_completeness_pct ASC;

In [0]:
WITH std AS (
  SELECT total_columns AS standard_column_count
  FROM metadata_governance.gold.table_governance_summary
  GROUP BY total_columns
  ORDER BY COUNT(*) DESC, total_columns DESC
  LIMIT 1
)
SELECT
  s.table_name              AS `Table`,
  s.system_name             AS `Source system`,
  s.total_columns           AS `Columns`,
  s.table_completeness_pct  AS `Completeness %`,
  s.maturity_tier           AS `Maturity`,
  s.pii_non_compliant_count AS `Sensitive data, no label`,
  s.unowned_count           AS `Columns with no owner`,
  s.uncertified_count       AS `Uncertified columns`,
  CASE WHEN s.total_columns = st.standard_column_count THEN 'Yes' ELSE 'No' END AS `Meets column standard`
FROM metadata_governance.gold.table_governance_summary s
CROSS JOIN std st
ORDER BY s.table_completeness_pct ASC, s.pii_non_compliant_count DESC
LIMIT 50;

In [0]:
SELECT * FROM (
  SELECT 'Sensitive data with no security label' AS exception_type,
         'High' AS severity,
         SUM(CASE WHEN pii_non_compliant THEN 1 ELSE 0 END) AS issue_count
  FROM metadata_governance.gold.column_governance_detail
  UNION ALL
  SELECT 'No data steward assigned', 'Medium',
         SUM(CASE WHEN unowned THEN 1 ELSE 0 END)
  FROM metadata_governance.gold.column_governance_detail
  UNION ALL
  SELECT 'Not certified', 'Low',
         SUM(CASE WHEN uncertified THEN 1 ELSE 0 END)
  FROM metadata_governance.gold.column_governance_detail
) AS ex
WHERE issue_count > 0
ORDER BY issue_count DESC;